In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv("dataset/predict-student-dropout-and-academic-success.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
print("Missing Values")
print(df.isnull().sum().sum())

print("\nDuplicates")
print(df.duplicated().sum())

print("\nTarget Distribution")
print(df["Target"].value_counts())

In [ ]:
# First Semester Approval Rate
df["Approval_Rate_1st"] = np.where(
    df["Curricular units 1st sem (enrolled)"] > 0,
    df["Curricular units 1st sem (approved)"] /
    df["Curricular units 1st sem (enrolled)"],
    0
)

# Second Semester Approval Rate
df["Approval_Rate_2nd"] = np.where(
    df["Curricular units 2nd sem (enrolled)"] > 0,
    df["Curricular units 2nd sem (approved)"] /
    df["Curricular units 2nd sem (enrolled)"],
    0
)

# Overall Approval Rate
df["Overall_Approval_Rate"] = np.where(
    (
        df["Curricular units 1st sem (enrolled)"] +
        df["Curricular units 2nd sem (enrolled)"]
    ) > 0,

    (
        df["Curricular units 1st sem (approved)"] +
        df["Curricular units 2nd sem (approved)"]
    ) /

    (
        df["Curricular units 1st sem (enrolled)"] +
        df["Curricular units 2nd sem (enrolled)"]
    ),

    0
)

In [ ]:
df["Failed_Units_1st"] = np.maximum(
    df["Curricular units 1st sem (enrolled)"] -
    df["Curricular units 1st sem (approved)"],
    0
)

df["Failed_Units_2nd"] = np.maximum(
    df["Curricular units 2nd sem (enrolled)"] -
    df["Curricular units 2nd sem (approved)"],
    0
)

In [ ]:
df["Average_Grade"] = (
    df["Curricular units 1st sem (grade)"] +
    df["Curricular units 2nd sem (grade)"]
) / 2

In [ ]:
df["Financial_Risk"] = np.where(
    (df["Debtor"] == 1) |
    (df["Tuition fees up to date"] == 0),
    1,
    0
)

In [ ]:
target_mapping = {
    "Graduate": 0,
    "Enrolled": 1,
    "Dropout": 2
}

df["Target"] = df["Target"].map(target_mapping)

In [ ]:
X = df.drop("Target", axis=1)

y = df["Target"]

In [ ]:
categorical_features = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

In [ ]:
numeric_features = [col for col in X.columns
                    if col not in categorical_features]

In [ ]:
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))

Train-Test Split, Stratified Sampling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

Numeric Pipeline

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

Categorical Pipeline

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)


Build Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Transforming the Dataset

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Train Shape:", X_train_processed.shape)
print("Test Shape:", X_test_processed.shape)

Saving the Preprocessor

In [ ]:
import joblib

joblib.dump(
    preprocessor,
    "dropout_preprocessor.pkl"
)

print("Preprocessor saved.")

----------------------------------After Preprocessing------------------------------

In [ ]:
#Logistic Regression Model
from sklearn.linear_model import LogisticRegression
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

import sys

print(sys.executable)

In [ ]:
!{sys.executable} -m pip install xgboost

In [ ]:
from xgboost import XGBClassifier


print("XGBoost installed successfully")

----------------------TRAINING OF BASELINE MODELS---------------------------------

In [ ]:
#RANDOM FOREST CLASSIFIER

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train_processed, y_train)

print("Random Forest trained successfully")

In [ ]:
#LOGISTIC REGRESSION MODEL
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

lr.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully")

In [ ]:
#XGBOOST CLASSIFIER
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train_processed, y_train)

print("XGBoost trained successfully")

--------------------------EVALUATION OF MODELS----------------------------------

In [ ]:

from sklearn.metrics import classification_report

In [ ]:
#RANDOM FOREST CLASSIFIER EVALUATION
rf_pred = rf.predict(X_test_processed)

print(classification_report(y_test, rf_pred))

In [ ]:
#LOGISTIC REGRESSION EVALUATION
lr_pred = lr.predict(X_test_processed)

print(classification_report(y_test, lr_pred))

In [ ]:
#XGBOOST evaluation
xgb_pred = xgb_model.predict(X_test_processed)

print(classification_report(y_test, xgb_pred))

In [ ]:
#COMPARISON TABLE

from sklearn.metrics import accuracy_score
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ]
})

results.sort_values(
    by="Accuracy",
    ascending=False
)


-----------------------CONFUSION MATRIX-----------------------

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

rf_pred = rf.predict(X_test_processed)

cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Graduate","Enrolled","Dropout"],
    yticklabels=["Graduate","Enrolled","Dropout"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")

plt.show()

------------------CLASSIFICATION REPORT--------------------------

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        rf_pred,
        target_names=[
            "Graduate",
            "Enrolled",
            "Dropout"
        ]
    )
)

--------------------FEATURE IMPORTANCE----------------
what factor influence dropout risk?


In [ ]:
feature_names = preprocessor.get_feature_names_out()

In [ ]:
import pandas as pd

importance_df = pd.DataFrame(
    {
        "Feature": feature_names,
        "Importance": rf.feature_importances_
    }
)

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

In [ ]:
importance_df.head(20)


In [ ]:
#Visualization of Feature Importance
plt.figure(figsize=(12,8))

sns.barplot(
    data=importance_df.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Important Features")
plt.show()

------------------HYPERPARAMETER TUNING-----------------------

In [ ]:
#PARAMETER GRID
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [200,300,500],
    'max_depth': [10,20,30,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'max_features': [
        'sqrt',
        'log2'
    ]
}

In [ ]:
#SEARCH

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),

    param_distributions=param_grid,

    n_iter=20,

    cv=5,

    scoring='f1_weighted',

    verbose=2,

    n_jobs=-1,

    random_state=42
)

In [ ]:
#TRAIN
rf_search.fit(
    X_train_processed,
    y_train
)

In [ ]:
#BEST PARAMETERS
print(
    rf_search.best_params_
)

print(
    rf_search.best_score_
)

In [ ]:
#Tuned MODEL
best_rf = rf_search.best_estimator_

In [ ]:
#Evaluation of tuned model
best_pred = best_rf.predict(
    X_test_processed
)

print(
    classification_report(
        y_test,
        best_pred
    )
)

---------------------------RISK LEVEL MAPPING-------------------------

In [ ]:
#Probability Predictions

probabilities = best_rf.predict_proba(
    X_test_processed
)

In [ ]:
#Checking Class Order

print(
    best_rf.classes_
)

In [ ]:
#Gettin dropout probability

dropout_prob = probabilities[:,2]

In [ ]:
#Risk Mapping Function

def risk_level(prob):

    if prob < 0.40:
        return "Low Risk"

    elif prob < 0.70:
        return "Medium Risk"

    else:
        return "High Risk"

In [ ]:
#Applying risk mapping function to dropout probabilities

risk_labels = [
    risk_level(p)
    for p in dropout_prob
]

In [ ]:
risk_df = pd.DataFrame(
    {
        "Actual": y_test.values,
        "Dropout_Probability": dropout_prob,
        "Risk_Level": risk_labels
    }
)

risk_df.head()

In [ ]:
#rf
print(classification_report(y_test, rf_pred))
#xgb
print(classification_report(y_test, xgb_pred))
#lr
print(classification_report(y_test, lr_pred))